# Class weighting for the flood occurrence classifier

## The problem

`dataset/flood_training_data.csv` holds **48,605 district-days with 121 floods** —
a ratio of about **400:1**, or 0.25% prevalence.

A model trained on ordinary loss can reach **99.75% accuracy by predicting "no
flood" every day**, and gradient descent will find exactly that: the 48,484
negatives collectively outweigh the 121 positives, so the model never receives a
strong enough signal that floods exist at all. Accuracy is a useless metric at
this prevalence.

## What class weighting is

Each sample's contribution to the loss is multiplied by a weight that depends on
its class. Misclassifying a flood day becomes expensive; misclassifying a dry day
stays cheap. **The data is not touched — only the loss function changes.** No rows
are duplicated, discarded or synthesised.

The conventional "balanced" weights make each class contribute equal total mass to
the loss, by weighting inversely to frequency:

$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_c}$$

so a class holding 1/400th of the rows gets ~400x the per-sample weight.

### The same idea has three different APIs

This is where the concept is easy but the implementation goes wrong silently:

| Framework | Argument | Expects |
| --- | --- | --- |
| **Keras / TensorFlow** | `model.fit(..., class_weight=...)` | a **dict** of both classes: `{0: w_neg, 1: w_pos}` |
| PyTorch | `BCEWithLogitsLoss(pos_weight=...)` | a **single scalar** — the ratio only |
| scikit-learn | `class_weight="balanced"` | computed internally |

Passing the ratio where Keras wants the pair, or the pair where PyTorch wants the
ratio, produces a wrong model with no error message. This notebook targets Keras,
so it emits the dict — and also prints the PyTorch scalar so the two are never
confused.

## What class weighting does *not* fix

Weighting changes emphasis, not information. Three consequences worth keeping in
view:

1. **It adds no data.** There are still only 121 flood examples, so variance stays
   high and overfitting is easy.
2. **It decalibrates the output probabilities.** Predictions will be inflated well
   above true flood likelihood. Treat them as ranking scores, not probabilities,
   unless recalibrated afterwards.
3. **A weight near 200 can destabilise training** — large gradient spikes and an
   oscillating loss. A damped weight is emitted below as a practical alternative;
   the weight is a hyperparameter, not a constant handed down by the formula.

## 1. Load and quantify the imbalance

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

TRAINING_DATA = "dataset/flood_training_data.csv"
SPLIT_OUT = "dataset/flood_training_data_split.csv"
WEIGHTS_OUT = "dataset/class_weights.json"

LABEL = "Flood occurrences"
TARGET_TRAIN_SHARE = 0.80     # of ROWS - see the note in section 2
N_CLASSES = 2


def find_repo_root(marker: str = "dataset") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()

data = pd.read_csv(REPO_ROOT / TRAINING_DATA, parse_dates=["date"])
data[LABEL] = data[LABEL].astype(bool)

positives = int(data[LABEL].sum())
negatives = len(data) - positives

print(f"rows        {len(data):,}")
print(f"positives   {positives}")
print(f"negatives   {negatives:,}")
print(f"ratio       {negatives / positives:.1f} : 1")
print(f"prevalence  {100 * positives / len(data):.3f}%")
print(f"\naccuracy from always predicting 'no flood': {100 * negatives / len(data):.2f}%")

## 2. The split has to come first

Weights must be computed from the **training split alone**. Deriving them from the
whole dataset lets the test set's class balance influence training — a mild form
of leakage, and one that makes the reported weights not quite the weights the
model was trained under.

### Why a random split would be invalid here

The 121 positive days are not 121 independent observations. They are **75 flood
runs**, 16 of which span more than one day:

A random row-level split would place day 3 of a flood in training and day 4 in
test. Those two rows share almost the same antecedent rainfall, the same district
and adjacent dates, so the model would "predict" the test row from having
memorised its neighbour. Scores would look excellent and mean nothing.

**The effective sample size is 75, not 121.**

### Chronological split

Training on the earlier period and testing on the later is leak-free by
construction and is the honest test for a forecasting problem — it asks whether
the model generalises *forward in time*, which is what deployment requires.

The boundary is chosen to put ~80% of positives in training **while not cutting
through any flood run**, verified rather than assumed.

In [ ]:
# A run is a maximal sequence of consecutive flood days in one district: the
# unit that must not be divided by the split
flood_days = data[data[LABEL]].sort_values(["district", "date"]).copy()
new_run = flood_days.groupby("district")["date"].diff().dt.days != 1
flood_days["run"] = new_run.cumsum()
runs = flood_days.groupby("run")["date"].agg(start="min", end="max")
run_lengths = flood_days.groupby("run").size()

print(f"{len(runs)} flood runs across {positives} positive days")
print("run length distribution:", run_lengths.value_counts().sort_index().to_dict())
print(f"runs longer than one day: {int((run_lengths > 1).sum())}")


def evaluate_boundary(boundary: pd.Timestamp) -> dict:
    """Row and positive counts either side, and how many runs would be cut.

    The target share is on ROWS, not positives. Targeting positives pushes the
    boundary very late, because positives cluster in the later years: aiming for
    80% of positives in training leaves a test set of only ~6% of rows whose
    prevalence is four times the training set's. Splitting on rows keeps the test
    set a usable size and, as it happens, yields more test positives too.
    """
    train_rows = int((data["date"] < boundary).sum())
    train_pos = int((flood_days["date"] < boundary).sum())
    cut = int(((runs["start"] < boundary) & (runs["end"] >= boundary)).sum())
    return {
        "boundary": boundary,
        "train_rows": train_rows,
        "train_positives": train_pos,
        "test_positives": positives - train_pos,
        "train_share": train_rows / len(data),
        "positive_share": train_pos / positives,
        "runs_cut": cut,
    }


# Month starts are natural candidates; only those cutting no run are eligible
candidates = pd.date_range(data["date"].min(), data["date"].max(), freq="MS")
options = pd.DataFrame([evaluate_boundary(b) for b in candidates])
eligible = options[(options["runs_cut"] == 0) & options["test_positives"].gt(0)].copy()
eligible["distance"] = (eligible["train_share"] - TARGET_TRAIN_SHARE).abs()

best = eligible.nsmallest(1, "distance").iloc[0]
BOUNDARY = best["boundary"]

print(f"\n{len(eligible)} eligible boundaries cut no run; "
      f"closest to a {TARGET_TRAIN_SHARE:.0%} share of rows:")
print(f"  {BOUNDARY:%Y-%m-%d}  {best['train_share']:.1%} of rows in training, "
      f"carrying {best['positive_share']:.1%} of positives "
      f"({int(best['train_positives'])} train / {int(best['test_positives'])} test)")

data["split"] = np.where(data["date"] < BOUNDARY, "train", "test")

# The integrity claim, asserted rather than trusted
straddling = flood_days.groupby("run")["date"].apply(
    lambda dates: (dates < BOUNDARY).any() and (dates >= BOUNDARY).any()
)
if straddling.any():
    raise ValueError(f"{int(straddling.sum())} flood runs straddle the split boundary")
print(f"\nno flood run straddles {BOUNDARY:%Y-%m-%d}: verified")

summary = data.groupby("split").agg(
    rows=("date", "size"),
    positives=(LABEL, "sum"),
    first=("date", "min"),
    last=("date", "max"),
)
summary["ratio"] = ((summary["rows"] - summary["positives"]) / summary["positives"]).round(1)
summary["prevalence_pct"] = (100 * summary["positives"] / summary["rows"]).round(3)
print()
print(summary.to_string())

## 3. Compute the weights

From the training split only.

In [ ]:
train = data[data["split"] == "train"]
train_pos = int(train[LABEL].sum())
train_neg = len(train) - train_pos
imbalance = train_neg / train_pos

# Balanced: each class contributes equal total mass to the loss
weight_negative = len(train) / (N_CLASSES * train_neg)
weight_positive = len(train) / (N_CLASSES * train_pos)

# Damped: sqrt keeps the direction while cutting the magnitude, which matters
# because a weight near 200 produces gradient spikes in a neural network
damped_positive = np.sqrt(imbalance)

weights = {
    "source": TRAINING_DATA,
    "split_boundary": f"{BOUNDARY:%Y-%m-%d}",
    "train_rows": len(train),
    "train_positives": train_pos,
    "train_negatives": train_neg,
    "imbalance_ratio": round(imbalance, 2),
    "keras_class_weight": {"0": round(weight_negative, 6), "1": round(weight_positive, 4)},
    "keras_class_weight_damped": {"0": 1.0, "1": round(damped_positive, 4)},
    "pytorch_pos_weight": round(imbalance, 2),
}

print(f"training split: {len(train):,} rows, {train_pos} positives, "
      f"{train_neg:,} negatives, {imbalance:.1f}:1\n")
print("balanced weights")
print(f"  negative (0)  {weight_negative:.4f}")
print(f"  positive (1)  {weight_positive:.2f}")
print(f"  ratio         {weight_positive / weight_negative:.1f}   (equals the imbalance, as it should)")
print(f"\ndamped alternative (sqrt of the imbalance)")
print(f"  negative (0)  1.0")
print(f"  positive (1)  {damped_positive:.2f}")
print(f"\nPyTorch equivalent, for reference only: pos_weight = {imbalance:.1f} (a scalar, not a pair)")

### Using these in Keras

```python
import json

with open("dataset/class_weights.json") as handle:
    weights = json.load(handle)

# Keys must be ints; JSON stringifies them
class_weight = {int(k): v for k, v in weights["keras_class_weight"].items()}

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    # Accuracy is meaningless at 0.25% prevalence - track PR-AUC instead
    metrics=[
        keras.metrics.AUC(curve="PR", name="pr_auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)

model.fit(X_train, y_train, class_weight=class_weight, ...)
```

Start with `keras_class_weight`. If the loss oscillates or the model collapses to
predicting everything positive, switch to `keras_class_weight_damped` — that is
the symptom a weight of ~200 produces.

**Evaluate with PR-AUC, and report precision and recall at whatever decision
threshold you choose.** Weighting shifts the effective threshold, so the default
0.5 cut-off is no longer meaningful; pick the threshold from the
precision-recall curve on the training split, never on the test split.

In [ ]:
data.to_csv(REPO_ROOT / SPLIT_OUT, index=False)
(REPO_ROOT / WEIGHTS_OUT).write_text(json.dumps(weights, indent=2) + "\n")

print(f"wrote {REPO_ROOT / SPLIT_OUT}")
print(f"  {len(data):,} rows, columns: {', '.join(data.columns)}")
print(f"wrote {REPO_ROOT / WEIGHTS_OUT}")
print()
print(json.dumps(weights, indent=2))

## Findings

**Balanced weights: `{0: 0.5011, 1: 233.73}`**, computed on the training split
alone — 38,799 rows with 83 positives, a 466.5:1 imbalance. The damped
alternative is `{0: 1.0, 1: 21.60}`.

### The split

Boundary **2014-03-01**, chosen from 244 candidates that cut no flood run, as the
closest to an 80% share of rows.

| Split | Rows | Positives | Prevalence | Ratio | Period |
| --- | --- | --- | --- | --- | --- |
| train | 38,799 (79.8%) | 83 | 0.214% | 466:1 | 1998-01-15 to 2014-02-28 |
| test | 9,806 (20.2%) | 38 | 0.388% | 257:1 | 2014-03-01 to 2018-06-20 |

**No flood run straddles the boundary** — asserted in code, not assumed.

**A note on how this boundary was chosen.** Targeting 80% of *positives* rather
than of rows produces a much worse split: because positives cluster in the later
years, it pushes the boundary to 2017-04 and leaves a test set of only 2,754 rows
(5.7%) whose prevalence is **four times** the training set's. Splitting on rows
gives a usable test set and, as it happens, more test positives (38 against 25).

### Three caveats to carry into training

1. **Test prevalence is 1.8x training prevalence** (0.388% against 0.214%). This is
   not fixable by moving the boundary — it is the reporting trend below. It means
   the test set is an *easier* population than the training set, so headline
   metrics will flatter the model slightly. Report training and test prevalence
   alongside any score.

2. **The two periods are not equally well labelled, and this is the biggest threat
   to the whole exercise.** Positives per five-year block: 1 (1998–99), 1
   (2000–04), 25 (2005–09), 62 (2010–14), 32 (2015–18). DesInventar reporting
   improved markedly over time; flooding did not increase 60-fold. So the training
   split's early years contribute roughly 13,000 negatives that plausibly contain
   unrecorded floods, teaching the model that 1998–2004 was nearly flood-free.
   **Worth testing a 2005-onward variant** — it costs 2 positives and removes the
   noisiest label period.

3. **The effective positive sample size is 75 runs, not 121 days** — 16 of the
   runs span multiple days. For the training split that is roughly 50 independent
   flood episodes against 8 candidate features. Regularise hard, expect wide
   run-to-run variance, and treat single-split results as indicative only. With 38
   test positives, one misclassification moves recall by 2.6 percentage points.

### And the thing class weighting will not do

It adds no information. These weights make the model *attend* to 83 flood
examples; they do not make there be more than 83. If the features cannot separate
the classes — and the threshold analysis found antecedent rainfall separates them
weakly, AUC 0.66–0.71, with temperature adding only ~0.4 °C of signal — weighting
will surface that honestly as low precision rather than hide it behind 99.75%
accuracy. That is the point of it.